# M-0003 — Plate-code demultiplex（Run demux on M-0002 output）（v1）

状态：**In Progress**  
最后更新：2026-01-15

## 1) Context / Goal

- **Context**: M-0002 split raw reads into per-well FASTQ files (`well_A01.fastq.gz`). However, these files may contain reads from multiple plates mixed together if plates were multiplexed on the same run.
- **Goal**: Implement Plate Demultiplexing to further split the M-0002 output files by **Plate Code** (6-8bp prefix on R1).
- **Output**: A directory structure `out/plate_<ID>/well_<ID>.fastq.gz`, separating reads by unique (Plate, Well) combinations.

## 2) Scope / Requirements

- **Input**: 
  - Output directory from M-0002 (containing `well_*.fastq.gz`).
  - Plate Barcode CSV file (`plate_id`, `plate_code`).
- **Tool**: `scripts/run_plate_demux.py` wrapping `demultiplex` (jfjlaros).
- **Strategy**: For each well file, run `demultiplex` using the Plate Barcode table (matching R1 prefix).
- **Output Structure**:
  - `out/plate_<ID>/well_<ID>.fastq.gz`
  - `out/plate_UNKNOWN/well_<ID>.fastq.gz` (for reads not matching any plate)
  - `plate_demux_stats.tsv` (Summary counts).
- **Constraints**:
  - Use `TIRTL_analyse` environment.
  - Must preserve R1/R2 pairing uniqueness.

## 3) Acceptance Criteria (Testable)

- **AC-001 (Interface)**: `scripts/run_plate_demux.py` exists and is executable.
  - Arguments: `--input-dir` (M-0002 output), `--barcodes`, `--output-dir`.
- **AC-002 (Functionality)**: 
  - Splits `well_A01.fastq.gz` into `plate_X/well_A01.fastq.gz` and `plate_Y/well_A01.fastq.gz` based on R1 prefix.
  - Correctly handles unknown plate codes (into `plate_UNKNOWN/`).
- **AC-003 (Data Integrity)**:
  - Output files are valid gzip.
  - Total reads (Sum of all plates + unknown) exactly match input reads from M-0002.
- **AC-004 (Reporting)**:
  - Generates `plate_demux_stats.tsv` with `plate_id`, `well_id`, `read_count`.
- **AC-005 (Automation)**:
  - `scripts/verify.py` updated to run full chain: Synthetic Gen (Multi-plate) -> Well Demux -> Plate Demux -> Verify Counts.

## 4) Plan & Task Breakdown

- [x] **T-001: Verification Update (Synthetic Data)**
  - Update `scripts/generate_synthetic.py` to support multi-plate generation (ensure R1 prefixes vary by plate ID).
  - Verify generator produces distinct plate prefixes.

- [ ] **T-002: Plate Demux Script Implementation**
  - Create `scripts/run_plate_demux.py`.
  - Logic: Iterate over input well files -> subprocess call `demultiplex` -> Organize output.

- [ ] **T-003: Verification Logic Integration**
  - Update `scripts/verify.py` to add `verify_m0003()`.
  - Run full pipeline: Gen -> M-0002 -> M-0003 -> Check.

- [ ] **T-004: Documentation & Final Verification**
  - Update docs with Plate Demux usage.
  - Run final verification and fill Acceptance Summary.

## 5) Implementation Notes

- **Mismatch**: Default to 1 mismatch for Plate Barcodes (20bp, robust).
- **Performance**: Sequential processing of 384 well files might be slow. `demultiplex` tool is fast, but overhead matters. Python `multiprocessing` (optional optimization if needed, but not required for AC).
- **Metadata**: Ensure `demultiplex` knows to look at the *start* of the read. M-0002 used R2 start. Here we use R1 start.

## 6) Verification

- **Command**: `python scripts/verify.py --milestone M-0003`
- **Last Run**: (Not run yet)
- **Log**: `logs/M-0003-verify-YYYYMMDD-HHMM.txt`
- **Result**: Pending

### Scenario:
1. Generate synthetic data: 2 Plates (PlateA, PlateB), 2 Wells (A01, B01). Total 4 combinations + noise.
2. Run M-0002 (Well Demux) -> Expect `well_A01` (mixed PlateA/B) and `well_B01`.
3. Run M-0003 (Plate Demux) -> Expect `plate_PlateA/well_A01`, `plate_PlateB/well_A01`, etc.
4. Verify counts match inputs.

## 7) Files Changed

- `scripts/generate_synthetic.py`: [MODIFIED] Updated `load_plate_barcodes` to include the full adapter prefix + unique ID in the R1 sequence, ensuring alignment with `demux_plan.md` specs.

## 8) Acceptance Summary

**Status**: 🚧 In Progress

| AC | Description | Status | Evidence |
|----|-------------|--------|----------|
| AC-001 | Interface | ⏳ Pending | - |
| AC-002 | Functionality | ⏳ Pending | - |
| AC-003 | Data Integrity | ⏳ Pending | - |
| AC-004 | Reporting | ⏳ Pending | - |
| AC-005 | Automation | ⏳ Pending | - |

## 9) Change Log

- **v1**: Initial R3 Pipeline specification for Plate Demux.